# FastqProcessor-Antibody optimized (FAO) Jupyternotebook version

# CELL1: IMPORTS and DEPENDENCIES
 Import the packages and dependencies need for FAO
 Run the cell and check if all necessary packages are installed
 Refer the TROUBLESHOOTING cell when error shows up

 # CELL2: PARAMETERS and CONFIGURATIONS 
 The only cell need to be edited
 Set the required parameters, run the cell and check if configurations are set well

 # CELL3: MAIN EXECUtion
 Run the cell to initiate the heavy lifting process. 
 Once they run this, you can step away and have a rest.

In [1]:
# ==========================================
# CELL 1: IMPORTS & DEPENDENCIES
# ==========================================
import os
import glob
import time
import multiprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Custom Pipeline Imports
from utils.ProcessHandlers import Pipeline, FastqParser, IgblastExtractor, EnrichmentAnalyzer
from utils.ProcessHandlers import Logger, DirectoryTracker

# Local Dispatcher Definition
class LocalDispatcher:
    def __init__(self, config_obj):
        self.cfg = config_obj
        
    def dispatch_handlers(self, handler_classes):
        l_conf = self.cfg.LoggerConfig
        t_conf = self.cfg.TrackerConfig
        shared_logger = Logger(l_conf).logger
        shared_dirs = DirectoryTracker(t_conf)
        
        instances = []
        for h_cls in handler_classes:
            if h_cls == Pipeline:
                instances.append(Pipeline({'exp_name': self.cfg.experiment, 'dirs': shared_dirs, 'logger': shared_logger, 'conf': None}))
            elif h_cls == FastqParser:
                instances.append(FastqParser({'dirs': shared_dirs, 'logger': shared_logger}))
            elif h_cls == IgblastExtractor:
                p_conf = self.cfg.ParserConfig
                meta = {
                    'dirs': shared_dirs, 'logger': shared_logger,
                    'igblast_exec': getattr(p_conf, 'igblast_exec', 'igblastn'),
                    'igdata_path': getattr(p_conf, 'igdata_path', ''),
                    'database_dir': getattr(p_conf, 'database_dir', ''),
                    'species': getattr(p_conf, 'species', 'human'),
                    'linker_seq': getattr(p_conf, 'linker_seq', ''),
                    'linker_tol_ratio': getattr(p_conf, 'linker_tol_ratio', 0.1),
                    'extract_regions': getattr(p_conf, 'extract_regions', ['FL','CDR3','CDRs']) 
                }
                instances.append(IgblastExtractor(meta))
            elif h_cls == EnrichmentAnalyzer:
                instances.append(EnrichmentAnalyzer(shared_dirs.parser_out, shared_logger))
                
        return tuple(instances)

print("✅ All dependencies loaded successfully.")

✅ All dependencies loaded successfully.


In [2]:
import os

# ==========================================
# CELL 2: PARAMETER CONFIGURATION
# ==========================================

# 1. Define base paths outside the class to avoid Python scoping errors
BASE_DIR = '/home/zhao/NGS/test'
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
OUT_DIR = os.path.join(BASE_DIR, 'parser_outputs')

class Config:
    experiment = 'Specifica_IgBLAST_Pipeline'
    
    class TrackerConfig:
        seq_data = BASE_DIR
        logs = LOGS_DIR
        parser_out = OUT_DIR

    class LoggerConfig:
        name = 'IgBLAST_screening_logger'
        verbose = True
        log_to_file = True
        log_fname = os.path.join(LOGS_DIR, name + '.log')

    class ParserConfig:
        igblast_exec = '/home/zhao/NGS/ncbi-igblast-1.22.0/bin/igblastn'
        igdata_path = '/home/zhao/NGS/ncbi-igblast-1.22.0'
        database_dir = '/home/zhao/NGS/ncbi-igblast-1.22.0/database'
        species = 'human' 
        linker_seq = 'TCCGGAGGGTCGACCATAACTTCGTATAATGTATACTATACGAAGTTATCCTCGAGCGGTACC'
        linker_tol_ratio = 0.1 
        extract_regions = ['CDR3', 'CDRs', 'FL']

    class FilterConfig:
        len_range = [200, 1200]
        min_q_score = 30
        q_pass_fraction = 0.9
        chunk_lines = 100000 
        delete_intermediate_chunks = True

    class AnalysisConfig:
        visualization_specs = ['H_CDR3_PEP', 'H_CDRs_PEP-L_CDRs_PEP', 'H_FL_PEP-L_FL_PEP']
        enrichment_specs = ['H_FL_PEP-L_FL_PEP']
        enrichment_power = 2.0
        retention_power = 1.0

# --- Print Summary for User Confirmation ---
print("="*50)
print(f"🧬 PIPELINE CONFIGURATION: {Config.experiment}")
print("="*50)
print(f"📂 Input Data:   {Config.TrackerConfig.seq_data}")
print(f"📂 Output Dir:   {Config.TrackerConfig.parser_out}")
print(f"🔬 Antibody species:      {Config.ParserConfig.species.upper()}")
print(f"✂️  Linker:       {Config.ParserConfig.linker_seq} (Tolerance: {Config.ParserConfig.linker_tol_ratio})")
print(f"🎯 Extractions:  {', '.join(Config.ParserConfig.extract_regions)}")
print(f"🛡️  QC Filters:   Length {Config.FilterConfig.len_range} | Q>{Config.FilterConfig.min_q_score} on {Config.FilterConfig.q_pass_fraction*100}% of read")
print("="*50)
print("Ready to execute Phase 1.")

🧬 PIPELINE CONFIGURATION: Specifica_IgBLAST_Pipeline
📂 Input Data:   /home/zhao/NGS/test
📂 Output Dir:   /home/zhao/NGS/test/parser_outputs
🔬 Antibody species:      HUMAN
✂️  Linker:       TCCGGAGGGTCGACCATAACTTCGTATAATGTATACTATACGAAGTTATCCTCGAGCGGTACC (Tolerance: 0.1)
🎯 Extractions:  CDR3, CDRs, FL
🛡️  QC Filters:   Length [200, 1200] | Q>30 on 90.0% of read
Ready to execute Phase 1.


In [ ]:
# ==========================================
# CELL 3: MAIN EXECUTION
# ==========================================

# 1. Initialize Dispatcher and Handlers
dispatcher = LocalDispatcher(Config)
pip, par, ig_extractor, enricher = dispatcher.dispatch_handlers(
    (Pipeline, FastqParser, IgblastExtractor, EnrichmentAnalyzer)
)

print("🚀 Starting Phase 1: Processing and Merging...")
# 2. Enqueue the sequence operations
pip.enque([
    par.len_filter(where='dna', len_range=Config.FilterConfig.len_range), 
    par.q_score_filt(minQ=Config.FilterConfig.min_q_score, frac=Config.FilterConfig.q_pass_fraction),
    ig_extractor.run_igblast_and_translate(), 
    par.count_summary(where='pep', fmt='csv'),
    par.unpad(),
])

# 3. Stream data and merge
data_iter = par.stream_chunks_from_gz_dir(chunk_lines=Config.FilterConfig.chunk_lines)
pip.run_over_stream(data_iter, save_summary=True)
pip.merge_chunk_outputs(delete_chunks=Config.FilterConfig.delete_intermediate_chunks)

print("✅ Phase 1 Complete. Starting Phase 2: Formatting Downstream CSVs...")
ig_extractor.format_for_downstream() 

print("✅ Phase 2 Complete. Starting Phase 3: Generating Convergence Plots...")
annotated_files = glob.glob(os.path.join(Config.TrackerConfig.parser_out, "*", "*_annotated.csv"))
for f_path in annotated_files:
    enricher.visualize_convergence(f_path, region_specs=Config.AnalysisConfig.visualization_specs)
    
print("✅ Phase 3 Complete. Starting Phase 4: Enrichment Analysis...")
enricher.auto_discover_and_analyze(
    region_specs=Config.AnalysisConfig.enrichment_specs,
    power=Config.AnalysisConfig.enrichment_power,
    retention_power=Config.AnalysisConfig.retention_power 
)

print("🎉 PIPELINE FULLY COMPLETE!")

[INFO]: The following handler was succesfully initialized: <utils.ProcessHandlers.Pipeline object at 0x7f2600f2a510>
[INFO]: The following handler was succesfully initialized: <utils.ProcessHandlers.FastqParser object at 0x7f2600f2a660>
[INFO]: 5 routines appended to pipeline.


🚀 Starting Phase 1: Processing and Merging...


[INFO]: --- Processing Chunk 1 (VHH_2A_R5__chunk1) ---
[INFO]: [INFO] VHH_2A_R5__chunk1: Count=4357 | DNA: MaxLen=1208 | PEP: Unknown | Q: MaxLen=1208
[INFO]: > Running len_filter_dna...
[INFO]: [INFO] VHH_2A_R5__chunk1: Count=4355 | DNA: MaxLen=1196 | PEP: Unknown | Q: MaxLen=1196
[INFO]: > Running q_score_filt_Q30...
[INFO]: [INFO] VHH_2A_R5__chunk1: Count=4295 | DNA: MaxLen=1196 | PEP: Unknown | Q: MaxLen=1196
[INFO]: > Running run_igblast_and_translate...
